# AgroMind — Climate-Smart Crop Advisor
### Fine-tuning Gemma 2B-IT with QLoRA + RAG Pipeline
**Covers all 7 evaluation criteria:**
1. Dataset quality, preprocessing, and proper data split
2. LLM Fine-tuning using PEFT (QLoRA)
3. Baseline comparison
4. Data storage (ChromaDB + SQLite)
5. Quantitative evaluation (BLEU, ROUGE)
6. Qualitative and error analysis
7. Real-world applicability

**Hardware:** RTX 3060 6GB | Python 3.11 | Windows

In [ ]:
import subprocess, sys

packages = [
    "torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121",
    "transformers==4.40.0 --no-deps",
    "peft==0.10.0",
    "trl==0.8.6",
    "bitsandbytes==0.46.1 --prefer-binary",
    "accelerate==1.6.0",
    "datasets==2.18.0",
    "scipy sentencepiece",
    "langchain langchain-community",
    "chromadb",
    "sentence-transformers",
    "evaluate rouge-score sacrebleu",
    "pandas==2.0.3",
    "numpy==1.26.4",
    "scikit-learn matplotlib",
    "huggingface_hub",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkg.split()])

print("\nVerifying:")
import bitsandbytes as bnb
import accelerate, transformers
print(f"bitsandbytes : {bnb.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"transformers : {transformers.__version__}")
print("\nAll done. Restart kernel now.")

KeyboardInterrupt: 

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Torch version: 2.5.1+cu121
Device: NVIDIA GeForce RTX 3060 Laptop GPU


## 0. Install Dependencies

In [ ]:
# Run this cell once, then restart kernel
import subprocess, sys

packages = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118",
    "transformers==4.40.0",
    "peft==0.10.0",
    "trl==0.8.6",
    "bitsandbytes==0.43.1 --prefer-binary",
    "datasets accelerate scipy sentencepiece",
    "langchain langchain-community chromadb sentence-transformers",
    "evaluate rouge-score sacrebleu",
    "pandas numpy scikit-learn",
    "kaggle"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *pkg.split()])

print("All packages installed. Restart kernel now.")

All packages installed. Restart kernel now.


## 1. GPU and Environment Check

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

try:
    import bitsandbytes as bnb
    print(f"bitsandbytes: {bnb.__version__} ✓")
except Exception as e:
    print(f"bitsandbytes error: {e}")

import transformers, peft, trl
print(f"transformers: {transformers.__version__}")
print(f"peft: {peft.__version__}")
print(f"trl: {trl.__version__}")

CUDA available: True
Device: NVIDIA GeForce RTX 3060 Laptop GPU
VRAM: 6.44 GB
bitsandbytes: 0.43.1 ✓


c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 4.40.0
peft: 0.10.0
trl: 0.8.6


## 2. Dataset Download and Loading
### CRITERION i — Dataset quality, preprocessing, proper data split

In [1]:
import pandas as pd

# Read directly from where it already is
df = pd.read_csv(r"C:\Users\ASUS\Desktop\Projects_endsem\Gen_Ai\data\Crop_recommendation.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head(3))

Shape: (2200, 8)
Columns: ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'label']
    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice


In [2]:
import os
import json
from sklearn.model_selection import train_test_split

os.makedirs("data/processed", exist_ok=True)

DATA_PATH = r"C:\Users\ASUS\Desktop\Projects_endsem\Gen_Ai\data\Crop_recommendation.csv"

SYSTEM_INSTRUCTION = (
    "You are AgroMind, a climate-smart crop advisor for Indian farmers. "
    "Based on the given soil nutrients and climate conditions, recommend "
    "the most suitable crop with clear reasoning and risk assessment."
)

def row_to_instruction(row):
    input_text = (
        f"N: {row['N']}, P: {row['P']}, K: {row['K']}, "
        f"Temperature: {round(row['temperature'], 2)}°C, "
        f"Humidity: {round(row['humidity'], 2)}%, "
        f"pH: {round(row['ph'], 2)}, "
        f"Rainfall: {round(row['rainfall'], 2)}mm"
    )

    crop = row["label"]

    risk_flags = []
    if row["rainfall"] < 50:   risk_flags.append("very low rainfall")
    if row["rainfall"] > 250:  risk_flags.append("excessive rainfall")
    if row["temperature"] > 38: risk_flags.append("high temperature stress")
    if row["ph"] < 5.5 or row["ph"] > 7.5: risk_flags.append("suboptimal soil pH")

    if len(risk_flags) == 0:
        risk, risk_reason = "Low", "All parameters are within optimal range."
    elif len(risk_flags) == 1:
        risk, risk_reason = "Medium", f"Moderate concern due to {risk_flags[0]}."
    else:
        risk, risk_reason = "High", f"Multiple stress factors: {', '.join(risk_flags)}."

    output_text = (
        f"Recommended crop: {crop.capitalize()}. "
        f"Reasoning: With N={row['N']}, P={row['P']}, K={row['K']}, "
        f"temperature of {round(row['temperature'],2)}°C, "
        f"humidity of {round(row['humidity'],2)}%, "
        f"pH of {round(row['ph'],2)}, and rainfall of {round(row['rainfall'],2)}mm, "
        f"{crop.capitalize()} is the most suitable crop. "
        f"Risk: {risk}. {risk_reason}"
    )

    return {
        "instruction": SYSTEM_INSTRUCTION,
        "input": input_text,
        "output": output_text
    }

# Convert all rows
instruction_data = [row_to_instruction(row) for _, row in df.iterrows()]

# Split 70 / 15 / 15
train_data, temp_data = train_test_split(instruction_data, test_size=0.30, random_state=42)
val_data, test_data   = train_test_split(temp_data, test_size=0.50, random_state=42)

print(f"Total pairs : {len(instruction_data)}")
print(f"Train       : {len(train_data)}")
print(f"Val         : {len(val_data)}")
print(f"Test        : {len(test_data)}")

# Save as JSONL
def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

save_jsonl(train_data, "data/processed/train.jsonl")
save_jsonl(val_data,   "data/processed/val.jsonl")
save_jsonl(test_data,  "data/processed/test.jsonl")

print("\nExample pair:")
print(json.dumps(instruction_data[0], indent=2))
print("\nSaved to data/processed/ ✓")

Total pairs : 2200
Train       : 1540
Val         : 330
Test        : 330

Example pair:
{
  "instruction": "You are AgroMind, a climate-smart crop advisor for Indian farmers. Based on the given soil nutrients and climate conditions, recommend the most suitable crop with clear reasoning and risk assessment.",
  "input": "N: 90, P: 42, K: 43, Temperature: 20.88\u00b0C, Humidity: 82.0%, pH: 6.5, Rainfall: 202.94mm",
  "output": "Recommended crop: Rice. Reasoning: With N=90, P=42, K=43, temperature of 20.88\u00b0C, humidity of 82.0%, pH of 6.5, and rainfall of 202.94mm, Rice is the most suitable crop. Risk: Low. All parameters are within optimal range."
}

Saved to data/processed/ ✓


In [3]:
# Dataset statistics — for your report
print("=== Dataset Statistics ===")
print(df.describe().round(2))
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

=== Dataset Statistics ===
             N        P        K  temperature  humidity       ph  rainfall
count  2200.00  2200.00  2200.00      2200.00   2200.00  2200.00   2200.00
mean     50.55    53.36    48.15        25.62     71.48     6.47    103.46
std      36.92    32.99    50.65         5.06     22.26     0.77     54.96
min       0.00     5.00     5.00         8.83     14.26     3.50     20.21
25%      21.00    28.00    20.00        22.77     60.26     5.97     64.55
50%      37.00    51.00    32.00        25.60     80.47     6.43     94.87
75%      84.25    68.00    49.00        28.56     89.95     6.92    124.27
max     140.00   145.00   205.00        43.68     99.98     9.94    298.56

Missing values: 0
Duplicates: 0


In [4]:
import json
import os
from sklearn.model_selection import train_test_split

os.makedirs("data/processed", exist_ok=True)

SYSTEM_INSTRUCTION = (
    "You are AgroMind, a climate-smart crop advisor for Indian farmers. "
    "Based on the given soil nutrients and climate conditions, recommend "
    "the most suitable crop with clear reasoning and risk assessment."
)

def row_to_instruction(row):
    input_text = (
        f"N: {row['N']}, "
        f"P: {row['P']}, "
        f"K: {row['K']}, "
        f"Temperature: {round(row['temperature'], 2)}°C, "
        f"Humidity: {round(row['humidity'], 2)}%, "
        f"pH: {round(row['ph'], 2)}, "
        f"Rainfall: {round(row['rainfall'], 2)}mm"
    )

    crop = row["label"]

    risk_flags = []
    if row["rainfall"] < 50:
        risk_flags.append("very low rainfall")
    if row["rainfall"] > 250:
        risk_flags.append("excessive rainfall")
    if row["temperature"] > 38:
        risk_flags.append("high temperature stress")
    if row["ph"] < 5.5 or row["ph"] > 7.5:
        risk_flags.append("suboptimal soil pH")

    if len(risk_flags) == 0:
        risk = "Low"
        risk_reason = "All soil and climate parameters are within optimal range."
    elif len(risk_flags) == 1:
        risk = "Medium"
        risk_reason = f"Moderate concern due to {risk_flags[0]}."
    else:
        risk = "High"
        risk_reason = f"Multiple stress factors: {', '.join(risk_flags)}."

    output_text = (
        f"Recommended crop: {crop.capitalize()}. "
        f"Reasoning: With N={row['N']}, P={row['P']}, K={row['K']}, "
        f"temperature of {round(row['temperature'], 2)}°C, "
        f"humidity of {round(row['humidity'], 2)}%, "
        f"soil pH of {round(row['ph'], 2)}, "
        f"and rainfall of {round(row['rainfall'], 2)}mm, "
        f"{crop.capitalize()} is the most suitable crop for these conditions. "
        f"Risk: {risk}. {risk_reason}"
    )

    return {
        "instruction": SYSTEM_INSTRUCTION,
        "input": input_text,
        "output": output_text
    }

# Convert all rows
instruction_data = [row_to_instruction(row) for _, row in df.iterrows()]

# Split 70 / 15 / 15
train_data, temp_data = train_test_split(instruction_data, test_size=0.30, random_state=42)
val_data, test_data   = train_test_split(temp_data, test_size=0.50, random_state=42)

print(f"Total pairs : {len(instruction_data)}")
print(f"Train       : {len(train_data)}")
print(f"Val         : {len(val_data)}")
print(f"Test        : {len(test_data)}")

# Save as JSONL
def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

save_jsonl(train_data, "data/processed/train.jsonl")
save_jsonl(val_data,   "data/processed/val.jsonl")
save_jsonl(test_data,  "data/processed/test.jsonl")

print("\nExample:")
print(json.dumps(instruction_data[0], indent=2))
print("\nSaved to data/processed/ ✓")

Total pairs : 2200
Train       : 1540
Val         : 330
Test        : 330

Example:
{
  "instruction": "You are AgroMind, a climate-smart crop advisor for Indian farmers. Based on the given soil nutrients and climate conditions, recommend the most suitable crop with clear reasoning and risk assessment.",
  "input": "N: 90, P: 42, K: 43, Temperature: 20.88\u00b0C, Humidity: 82.0%, pH: 6.5, Rainfall: 202.94mm",
  "output": "Recommended crop: Rice. Reasoning: With N=90, P=42, K=43, temperature of 20.88\u00b0C, humidity of 82.0%, soil pH of 6.5, and rainfall of 202.94mm, Rice is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range."
}

Saved to data/processed/ ✓


In [5]:
# Train / Val / Test split — 70 / 15 / 15
train_data, temp_data = train_test_split(instruction_data, test_size=0.30, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

# Save splits
def save_jsonl(data, path):
    with open(path, "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")

save_jsonl(train_data, "data/processed/train.jsonl")
save_jsonl(val_data,   "data/processed/val.jsonl")
save_jsonl(test_data,  "data/processed/test.jsonl")

print("Splits saved to data/processed/")

Train: 1540 | Val: 330 | Test: 330
Splits saved to data/processed/


In [6]:
import bitsandbytes, transformers, accelerate
print("bitsandbytes:", bitsandbytes.__version__)
print("transformers:", transformers.__version__)
print("accelerate:  ", accelerate.__version__)

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bitsandbytes: 0.46.1
transformers: 4.40.0
accelerate:   0.27.2


## 3. Baseline 1 — Zero-shot Gemma 2B-IT (No RAG, No Fine-tuning)
### CRITERION iii — Baseline comparison

In [ ]:
import torch, gc
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig



# Clear VRAM first
try:
    del baseline_model
except: pass
try:
    del model
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")


print("Logged in ✓")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading Gemma 2B-IT for baseline...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
baseline_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
baseline_model.eval()
print("Baseline model loaded ✓")
print(f"VRAM used: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

VRAM free: 6.44 GB
Token has not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to C:\Users\ASUS\.cache\huggingface\token
Login successful
Logged in ✓
Loading Gemma 2B-IT for baseline...


Gemma's activation function should be approximate GeLU and not exact GeLU.
Changing the activation function to `gelu_pytorch_tanh`.if you want to use the legacy `gelu`, edit the `model.config` to set `hidden_activation=gelu`   instead of `hidden_act`. See https://github.com/huggingface/transformers/pull/29402 for more details.
Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.90s/it]


Baseline model loaded ✓
VRAM used: 2.07 GB


In [ ]:
import subprocess, sys

# Check which pip is being used
print("Python:", sys.executable)


Python: c:\Users\ASUS\AppData\Local\Programs\Python\Python310\python.exe
nsformers]
   -------------------- ------------------- 1/2 [accelerate]
   -------------------- ------------------- 1/2 [accelerate]
   -------------------- ------------------- 1/2 [accelerate]
   -------------------- ------------------- 1/2 [accelerate]
   -------------------- ------------------- 1/2 [accelerate]
   -------------------- ------------------- 1/2 [accelerate]
   ---------------------------------------- 2/2 [accelerate]


 -orch (c:\users\asus\appdata\local\programs\python\python310\lib\site-packages)



In [ ]:
import bitsandbytes, transformers, accelerate
print("bitsandbytes:", bitsandbytes.__version__)
print("transformers:", transformers.__version__)
print("accelerate:  ", accelerate.__version__)
print("\nbitsandbytes location:", bitsandbytes.__file__)
print("transformers location:", transformers.__file__)

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


bitsandbytes: 0.46.1
transformers: 4.40.0
accelerate:   0.27.2

bitsandbytes location: c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\bitsandbytes\__init__.py
transformers location: c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\__init__.py


In [ ]:
def generate_response(model, tokenizer, instruction, input_text, max_new_tokens=200):
    prompt = f"""<start_of_turn>user
{instruction}

{input_text}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only model response
    if "model" in decoded:
        return decoded.split("model")[-1].strip()
    return decoded.strip()

# Run baseline on first 50 test samples
baseline_outputs = []
test_subset = test_data[:50]

print("Running Baseline 1 (zero-shot, no RAG, no fine-tuning)...")
for i, sample in enumerate(test_subset):
    output = generate_response(
        baseline_model, tokenizer,
        sample["instruction"], sample["input"]
    )
    baseline_outputs.append({
        "input": sample["input"],
        "reference": sample["output"],
        "baseline1_output": output
    })
    if i % 10 == 0:
        print(f"  {i+1}/50 done")

print("\nBaseline 1 example:")
print("INPUT:", baseline_outputs[0]["input"])
print("REFERENCE:", baseline_outputs[0]["reference"])
print("BASELINE OUTPUT:", baseline_outputs[0]["baseline1_output"])

Running Baseline 1 (zero-shot, no RAG, no fine-tuning)...
  1/50 done
  11/50 done
  21/50 done
  31/50 done
  41/50 done

Baseline 1 example:
INPUT: N: 14, P: 19, K: 14, Temperature: 17.68°C, Humidity: 94.36%, pH: 6.7, Rainfall: 108.06mm
REFERENCE: Recommended crop: Orange. Reasoning: With N=14, P=19, K=14, temperature of 17.68°C, humidity of 94.36%, soil pH of 6.7, and rainfall of 108.06mm, Orange is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range.
BASELINE OUTPUT: ## Recommended Crop:

Based on the given soil nutrients and climate conditions, the most suitable crop for AgroMind is **Rice (Oryza sativa)**.

**Reasoning:**

* **High Nitrogen (N):** Rice requires a significant amount of nitrogen for optimal growth. The given N value of 14 indicates a moderate nitrogen requirement.
* **High Phosphorus (P):** Rice also requires phosphorus for maximum yield. The given P value of 19 suggests a high phosphorus requirement.
* *

In [ ]:
# Free GPU memory before fine-tuning
import gc
del baseline_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed. Available: {torch.cuda.memory_reserved(0)/1e9:.2f} GB")

VRAM freed. Available: 0.00 GB


In [ ]:
import numpy, pandas, transformers, accelerate, bitsandbytes
print("numpy:       ", numpy.__version__)
print("pandas:      ", pandas.__version__)
print("transformers:", transformers.__version__)
print("accelerate:  ", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


numpy:        1.26.4
pandas:       2.2.2
transformers: 4.40.0
accelerate:   0.27.2
bitsandbytes: 0.43.1


## 4. QLoRA Fine-tuning on Gemma 2B-IT
### CRITERION ii — Effective LLM Fine-tuning using PEFT

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer
from datasets import Dataset
import torch, os

MODEL_ID = "google/gemma-2b-it"
OUTPUT_DIR = "./agromind_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 4-bit quantization config — tuned for 6GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # float16 for RTX 3060
    bnb_4bit_use_double_quant=True,        # saves ~0.4GB extra
)

print("Loading Gemma 2B-IT in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)
print("Model loaded ✓")

Loading Gemma 2B-IT in 4-bit...


Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.48s/it]


Model loaded ✓


In [ ]:
# LoRA configuration
# Justification: QLoRA freezes 98% of weights, trains only low-rank adapter
# matrices on attention layers. Reduces memory from ~16GB (full fine-tune)
# to ~5GB, making it feasible on a 6GB RTX 3060.

lora_config = LoraConfig(
    r=16,                      # rank — balance between quality and memory
    lora_alpha=32,             # scaling factor (alpha = 2*r is standard)
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Trainable parameters: 19,611,648 / 2,525,784,064 (0.78%)


In [ ]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("data/processed/train.jsonl")
val_data   = load_jsonl("data/processed/val.jsonl")
test_data  = load_jsonl("data/processed/test.jsonl")

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")

Train: 1540 | Val: 330 | Test: 330


In [ ]:
from datasets import Dataset
# Format dataset for SFTTrainer
def format_prompt(sample):
    return {
        "text": f"""<start_of_turn>user
{sample['instruction']}

{sample['input']}<end_of_turn>
<start_of_turn>model
{sample['output']}<end_of_turn>"""
    }

train_dataset = Dataset.from_list(train_data).map(format_prompt)
val_dataset   = Dataset.from_list(val_data).map(format_prompt)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples:   {len(val_dataset)}")
print("\nExample formatted prompt:")
print(train_dataset[0]["text"])

Map: 100%|██████████| 330/330 [00:00<00:00, 23569.12 examples/s]

Train samples: 1540
Val samples:   330

Example formatted prompt:
<start_of_turn>user
You are AgroMind, a climate-smart crop advisor for Indian farmers. Based on the given soil nutrients and climate conditions, recommend the most suitable crop with clear reasoning and risk assessment.

N: 21, P: 26, K: 27, Temperature: 27.0°C, Humidity: 47.68%, pH: 5.7, Rainfall: 95.85mm<end_of_turn>
<start_of_turn>model
Recommended crop: Mango. Reasoning: With N=21, P=26, K=27, temperature of 27.0°C, humidity of 47.68%, soil pH of 5.7, and rainfall of 95.85mm, Mango is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range.<end_of_turn>


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

OUTPUT_DIR = "./agromind_finetuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=256,
    tokenizer=tokenizer,
    args=training_args,
)

print("Trainer configured ✓")
print(f"Steps per epoch: {len(train_dataset) // 8}")
print(f"Total steps: {len(train_dataset) // 8 * 3}")

Map: 100%|██████████| 330/330 [00:00<00:00, 9033.19 examples/s]


Trainer configured ✓
Steps per epoch: 192
Total steps: 576


c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\accelerate\accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
# START FINE-TUNING
# Expected time on RTX 3060 6GB: ~2-3 hours for 3 epochs
print("Starting fine-tuning...")
print(f"VRAM before training: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

trainer.train()

print("\nFine-tuning complete ✓")

Starting fine-tuning...
VRAM before training: 5.27 GB


  0%|          | 0/576 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
  4%|▍         | 25/576 [07:20<2:38:27, 17.26s/it]

{'loss': 1.6651, 'grad_norm': 0.9225860834121704, 'learning_rate': 0.0001999223500431082, 'epoch': 0.13}


  9%|▊         | 50/576 [14:30<2:33:50, 17.55s/it]

{'loss': 0.3738, 'grad_norm': 0.3737246096134186, 'learning_rate': 0.0001983814494392829, 'epoch': 0.26}


...

{'loss': 0.2129, 'grad_norm': 0.2976922392845154, 'learning_rate': 1.0694793674208114e-06, 'epoch': 2.86}


100%|█████████▉| 575/576 [8:55:22<01:04, 64.67s/it]

{'loss': 0.2111, 'grad_norm': 0.2315622717142105, 'learning_rate': 1.5848939393436902e-09, 'epoch': 2.99}


100%|██████████| 576/576 [8:56:26<00:00, 55.88s/it]

{'train_runtime': 32186.5657, 'train_samples_per_second': 0.144, 'train_steps_per_second': 0.018, 'train_loss': 0.29986109778595466, 'epoch': 2.99}

Fine-tuning complete ✓


In [ ]:
# Save LoRA adapter weights
ADAPTER_PATH = "./agromind_adapter"
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter weights saved to {ADAPTER_PATH}")
print("Files:", os.listdir(ADAPTER_PATH))

LoRA adapter weights saved to ./agromind_adapter
Files: ['adapter_config.json', 'adapter_model.safetensors', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']


In [ ]:
# Optional: Merge adapter into base model and save full model
# WARNING: Requires ~12GB RAM. Skip if RAM is limited.
from peft import PeftModel

MERGED_PATH = "./agromind_merged"
os.makedirs(MERGED_PATH, exist_ok=True)

print("Merging adapter with base model...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(MERGED_PATH)
tokenizer.save_pretrained(MERGED_PATH)
print(f"Merged model saved to {MERGED_PATH}")

## 5. Load Fine-tuned Model for Inference

In [ ]:
#CLEAR RAM
import torch, gc

try:
    del model
except: pass
try:
    del trainer
except: pass
try:
    del baseline_model
except: pass

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

VRAM free: 6.42 GB


In [ ]:
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID    = "google/gemma-2b-it"
ADAPTER_PATH = "./agromind_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
ft_model.eval()
print("Fine-tuned model loaded ✓")

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.32s/it]


Fine-tuned model loaded ✓


## 6. RAG Pipeline — ChromaDB Vector Store
### CRITERION iv — Data Storage (Vector DB)

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer
import pandas as pd

os.makedirs("chromadb_store", exist_ok=True)

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="./chromadb_store")
collection = chroma_client.get_or_create_collection(
    name="agromind_knowledge",
    metadata={"hnsw:space": "cosine"}
)

# Embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("ChromaDB initialized ✓")
print("Embedder loaded ✓")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


ChromaDB initialized ✓
Embedder loaded ✓


In [ ]:
# Build knowledge base from the dataset itself
# Each crop's average profile becomes a retrievable knowledge chunk

df = pd.read_csv("data/Crop_recommendation.csv")
crop_profiles = df.groupby("label").mean().round(2)

documents = []
doc_ids   = []
metadatas = []

for crop, row in crop_profiles.iterrows():
    doc = (
        f"{crop.capitalize()} grows best with the following conditions: "
        f"Nitrogen (N) around {row['N']} kg/ha, "
        f"Phosphorus (P) around {row['P']} kg/ha, "
        f"Potassium (K) around {row['K']} kg/ha, "
        f"temperature between {row['temperature']-3:.1f}°C and {row['temperature']+3:.1f}°C, "
        f"humidity around {row['humidity']:.1f}%, "
        f"soil pH of {row['ph']:.2f}, "
        f"and rainfall of approximately {row['rainfall']:.1f}mm."
    )
    documents.append(doc)
    doc_ids.append(f"crop_{crop}")
    metadatas.append({"crop": crop})

# Add additional knowledge chunks about soil types and seasons
extra_knowledge = [
    ("sandy_loam",   "Sandy loam soil has good drainage and is suitable for crops like bajra, moong, groundnut, and cotton. It has lower water retention and requires more frequent irrigation in dry seasons."),
    ("clay_soil",    "Clay soil retains moisture well and is suitable for rice, wheat, and sugarcane. It can become waterlogged in high rainfall areas and may restrict root growth."),
    ("loamy_soil",   "Loamy soil is the most fertile and well-balanced, suitable for a wide range of crops including maize, cotton, vegetables, and pulses."),
    ("kharif",       "Kharif season in India runs from June to October. Major kharif crops include rice, maize, bajra, cotton, soybean, and groundnut. These crops require warm temperatures and monsoon rainfall."),
    ("rabi",         "Rabi season in India runs from November to April. Major rabi crops include wheat, barley, mustard, peas, and chickpea. These crops prefer cooler temperatures and minimal rainfall."),
    ("zaid",         "Zaid season in India runs from March to June. Major zaid crops include watermelon, muskmelon, cucumber, and moong. These crops need hot and dry weather."),
    ("rajasthan",    "Rajasthan is an arid to semi-arid region. Suitable crops include bajra, jowar, moong, moth bean, guar, and mustard. Water availability is the primary constraint. Rainfall averages 300-600mm annually."),
    ("punjab",       "Punjab has fertile alluvial soil and good irrigation. Major crops are wheat, rice, maize, and cotton. Average rainfall is 500-800mm. Double cropping is common."),
]

for kid, text in extra_knowledge:
    documents.append(text)
    doc_ids.append(f"know_{kid}")
    metadatas.append({"crop": "general"})

# Embed and store
embeddings = embedder.encode(documents, show_progress_bar=True).tolist()
collection.upsert(
    documents=documents,
    embeddings=embeddings,
    ids=doc_ids,
    metadatas=metadatas
)

print(f"\nChromaDB populated with {collection.count()} documents ✓")

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.09it/s]


ChromaDB populated with 30 documents ✓


In [ ]:
# RAG retrieval function
def retrieve_context(query, n_results=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    return "\n".join(results["documents"][0])

# Test retrieval
test_query = "sandy loam soil, low rainfall, high temperature, Rajasthan"
context = retrieve_context(test_query)
print("Retrieved context for test query:")
print(context)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Retrieved context for test query:
Rajasthan is an arid to semi-arid region. Suitable crops include bajra, jowar, moong, moth bean, guar, and mustard. Water availability is the primary constraint. Rainfall averages 300-600mm annually.
Sandy loam soil has good drainage and is suitable for crops like bajra, moong, groundnut, and cotton. It has lower water retention and requires more frequent irrigation in dry seasons.
Loamy soil is the most fertile and well-balanced, suitable for a wide range of crops including maize, cotton, vegetables, and pulses.


## 7. SQLite Query Logger
### CRITERION iv — Data Storage (Regular DB)

In [ ]:
import sqlite3
from datetime import datetime

def init_db(db_path="agromind_logs.db"):
    conn = sqlite3.connect(db_path)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS query_logs (
            id          INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp   TEXT,
            input_text  TEXT,
            context     TEXT,
            output      TEXT,
            model_type  TEXT
        )
    """)
    conn.commit()
    return conn

def log_query(conn, input_text, context, output, model_type):
    conn.execute("""
        INSERT INTO query_logs (timestamp, input_text, context, output, model_type)
        VALUES (?, ?, ?, ?, ?)
    """, (datetime.now().isoformat(), input_text, context, output, model_type))
    conn.commit()

db_conn = init_db()
print("SQLite logger initialised ✓")

SQLite logger initialised ✓


## 8. Full RAG + Fine-tuned Model Inference Pipeline

In [ ]:
def agromind_inference(input_text, model, tokenizer):
    prompt = f"""<start_of_turn>user
You are AgroMind, a crop advisor for Indian farmers. Recommend the most suitable crop with reasoning and risk assessment.

{input_text}<end_of_turn>
<start_of_turn>model
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Cut everything after first end of response
    for stop in ["model", "<end_of_turn>", "\nRecommended", "\nziua"]:
        if stop in response:
            response = response[:response.index(stop)].strip()

    return response

# Test
test_input = "N: 20, P: 30, K: 40, Temperature: 35.0°C, Humidity: 65.0%, pH: 7.2, Rainfall: 45.0mm"
result = agromind_inference(test_input, ft_model, tokenizer)
print("OUTPUT:")
print(result)

OUTPUT:
Recommended crop: Lentil. Reasoning: With N=20, P=30, K=40, temperature of 35.0°C, humidity of 65.0%, soil pH of 7.2, and rainfall of 189.0mm, Lentil is the most suitable crop for these conditions. Risk: Medium. Moderate concern due to very low rainfall.


In [ ]:
test_cases = [
    "N: 90, P: 42, K: 43, Temperature: 20.88°C, Humidity: 82.0%, pH: 6.5, Rainfall: 202.94mm",
    "N: 60, P: 55, K: 44, Temperature: 23.0°C, Humidity: 82.0%, pH: 7.8, Rainfall: 263.96mm",
    "N: 20, P: 15, K: 20, Temperature: 38.0°C, Humidity: 90.0%, pH: 4.5, Rainfall: 10.0mm",
]

for i, t in enumerate(test_cases):
    result = agromind_inference(t, ft_model, tokenizer)
    print(f"\nTest {i+1}:")
    print(f"Input: {t}")
    print(f"Output: {result}")
    print("-"*60)


Test 1:
Input: N: 90, P: 42, K: 43, Temperature: 20.88°C, Humidity: 82.0%, pH: 6.5, Rainfall: 202.94mm
Output: Recommended crop: Rice. Reasoning: With N=90, P=42, K=43, temperature of 20.88°C, humidity of 82.0%, soil pH of 7.1, and rainfall of 202.94mm, Rice is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range.
------------------------------------------------------------

Test 2:
Input: N: 60, P: 55, K: 44, Temperature: 23.0°C, Humidity: 82.0%, pH: 7.8, Rainfall: 263.96mm
Output: Recommended crop: Rice. Reasoning: With N=60, P=55, K=44, temperature of 23.0°C, humidity of 82.0%, soil pH of 7.8, and rainfall of 199.96mm, Rice is the most suitable crop for these conditions. Risk: Medium. Moderate concern due to excessive rainfall.
------------------------------------------------------------

Test 3:
Input: N: 20, P: 15, K: 20, Temperature: 38.0°C, Humidity: 90.0%, pH: 4.5, Rainfall: 10.0mm
Output: Recommended crop: Muskmelon.

## 9. Baseline 2 — Base Model + RAG (No Fine-tuning)

In [ ]:
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_data = load_jsonl("data/processed/test.jsonl")
test_subset = test_data[:50]
print(f"Test subset: {len(test_subset)} samples")
print("Example:", test_subset[0]["input"])

Test subset: 50 samples
Example: N: 14, P: 19, K: 14, Temperature: 17.68°C, Humidity: 94.36%, pH: 6.7, Rainfall: 108.06mm


In [15]:
import gc, torch, json

# Clear ft_model from VRAM
del ft_model
del base_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

NameError: name 'ft_model' is not defined

In [16]:
import gc, torch

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

VRAM free: 4.36 GB


In [ ]:
import gc, torch

# Kill everything
for var in ['ft_model', 'base_model', 'model', 'baseline1_model', 
            'base_model_b2', 'baseline_model', 'trainer']:
    try:
        exec(f'del {var}')
    except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"VRAM used: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

VRAM used: 0.01 GB
VRAM free: 6.43 GB


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch, json, gc

HF_TOKEN = "REPLACE_WITH_YOUR_HF_TOKEN"
MODEL_ID  = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

b1_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
baseline1_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
baseline1_model.eval()
print("Baseline 1 loaded ✓")

def baseline_inference(input_text, model, tokenizer):
    prompt = f"""<start_of_turn>user
You are a crop advisor. Recommend the most suitable crop with reasoning and risk assessment.

{input_text}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    for stop in ["model", "<end_of_turn>", "\nRecommended"]:
        if stop in response:
            response = response[:response.index(stop)].strip()
    return response

baseline1_outputs = []
print("Running Baseline 1 (zero-shot, no RAG, no FT)...")
for i, sample in enumerate(test_subset):
    output = baseline_inference(sample["input"], baseline1_model, b1_tokenizer)
    baseline1_outputs.append({
        "input":     sample["input"],
        "reference": sample["output"],
        "b1_output": output
    })
    if i % 10 == 0:
        print(f"  {i+1}/50 done")

with open("baseline1_outputs.json", "w") as f:
    json.dump(baseline1_outputs, f, indent=2)
print("Baseline 1 saved ✓")

del baseline1_model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.40s/it]


Baseline 1 loaded ✓
Running Baseline 1 (zero-shot, no RAG, no FT)...
  1/50 done
  11/50 done
  21/50 done
  31/50 done
  41/50 done
Baseline 1 saved ✓
VRAM free: 6.43 GB


In [ ]:
import json, torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


ADAPTER_PATH = "./agromind_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def infer(input_text, model, tokenizer):
    prompt = f"""<start_of_turn>user
You are AgroMind, a crop advisor for Indian farmers. Recommend the most suitable crop with reasoning and risk assessment.

{input_text}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.5,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    for stop in ["model", "<end_of_turn>", "\nRecommended"]:
        if stop in response:
            response = response[:response.index(stop)].strip()
    return response

# Load test subset
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_data   = load_jsonl("data/processed/test.jsonl")
test_subset = test_data[:50]

# Load fine-tuned model
print("Loading fine-tuned model...")
gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    device_map="auto", token=HF_TOKEN)
ft_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
ft_model.eval()
print(f"FT model loaded. VRAM: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")

ft_outputs = []
print("Running fine-tuned model on 50 samples...")
for i, sample in enumerate(test_subset):
    out = infer(sample["input"], ft_model, tokenizer)
    ft_outputs.append({
        "input":     sample["input"],
        "reference": sample["output"],
        "ft_output": out
    })
    if i % 10 == 0:
        print(f"  {i+1}/50")

with open("ft_outputs.json", "w") as f:
    json.dump(ft_outputs, f, indent=2)
print("Fine-tuned outputs saved ✓")

del ft_model, base
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0))/1e9:.2f} GB")
print("Done ✓")

Loading fine-tuned model...


Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.84s/it]


FT model loaded. VRAM: 2.12 GB
Running fine-tuned model on 50 samples...
  1/50
  11/50
  21/50
  31/50
  41/50
Fine-tuned outputs saved ✓
VRAM free: 6.43 GB
Done ✓


In [ ]:
import json, torch, gc
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel


ADAPTER_PATH = "./agromind_adapter"


def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

test_data   = load_jsonl("data/processed/test.jsonl")
test_subset = test_data[:50]
print(f"Test subset: {len(test_subset)}")

Token has not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to C:\Users\ASUS\.cache\huggingface\token
Login successful
Test subset: 50


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer fresh from base — do NOT set pad_token
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
print(f"pad: {tokenizer.pad_token_id} eos: {tokenizer.eos_token_id}")

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config,
    
ft_model = PeftModel.from_pretrained(base, ADAPTER_PATH)
ft_model.eval()
print("Loaded ✓")

pad: 0 eos: 1


Gemma's activation function should be approximate GeLU and not exact GeLU.
Changing the activation function to `gelu_pytorch_tanh`.if you want to use the legacy `gelu`, edit the `model.config` to set `hidden_activation=gelu`   instead of `hidden_act`. See https://github.com/huggingface/transformers/pull/29402 for more details.
Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.09s/it]


Loaded ✓


In [9]:
prompt = f"""<start_of_turn>user
You are AgroMind, a crop advisor for Indian farmers. Recommend the most suitable crop with reasoning and risk assessment.

{test_subset[0]["input"]}<end_of_turn>
<start_of_turn>model
"""

inputs = tokenizer(prompt, return_tensors="pt").to(ft_model.device)
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.3,
    )

new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

Recommended crop: Orange. Reasoning: With N=14, P=19, K=14, temperature of 17.68°C, humidity of 94.36%, soil pH of 6.7, and rainfall of 108.06mm, Orange is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range.model
 siguranta. Recommended crop: Orange. Reasoning: With N=14, P=19, K=14, temperature of 17.68°C, humidity of 94.36%, soil pH of 6.7, and rainfall of 10


In [10]:
def infer(input_text, model, tokenizer):
    prompt = f"""<start_of_turn>user
You are AgroMind, a crop advisor for Indian farmers. Recommend the most suitable crop with reasoning and risk assessment.

{input_text}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.3,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    # Cut at repetition or garbage
    for stop in ["model\n", "\nmodel", "siguranta", "ziua"]:
        if stop in response:
            response = response[:response.index(stop)].strip()
    
    return response

# Test
test_out = infer(test_subset[0]["input"], ft_model, tokenizer)
print("OUTPUT:")
print(test_out)
print(f"Length: {len(test_out)}")

OUTPUT:
Recommended crop: Orange. Reasoning: With N=14, P=19, K=14, temperature of 17.68°C, humidity of 94.36%, soil pH of 6.7, and rainfall of 108.06mm, Orange is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range.
Length: 269


## 10. Quantitative Evaluation — BLEU, ROUGE
### CRITERION v — Quantitative Performance Evaluation

In [11]:
inputs = tokenizer(prompt, return_tensors="pt").to(ft_model.device)
with torch.no_grad():
    outputs = ft_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.3,
        repetition_penalty=1.3,
        pad_token_id=0,
        eos_token_id=1,
    )

new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

Recommended crop: Orange. Reasoning: With N=14, P=19, K=14, temperature of 17.68°C, humidity of 94.36%, soil pH of 6.7, and rainfall of 108.06mm, Orange is the most suitable crop for these conditions. Risk: Low. All soil and climate parameters are within optimal range.model
 picioare. Most suitable crop: Orange. Reasoning: With N=14, P=19, K=14, temperature of 17.68°C, humidity of 94.36%, soil pH of 6.7, and rainfall of 10


In [12]:
import pandas as pd

results_df = pd.DataFrame([
    {"Model": "Baseline 1 (zero-shot)",    **metrics_b1},
    {"Model": "Baseline 2 (RAG only)",     **metrics_b2},
    {"Model": "AgroMind (FT + RAG)",       **metrics_ft},
])

print("\n=== FULL COMPARISON TABLE ===")
print(results_df.to_string(index=False))

results_df.to_csv("evaluation_results.csv", index=False)
print("\nSaved to evaluation_results.csv")

NameError: name 'metrics_b1' is not defined

## 11. Qualitative and Error Analysis
### CRITERION vi — Hallucination and Failure Cases

In [13]:
KNOWN_CROPS = [
    "rice", "maize", "chickpea", "kidneybeans", "pigeonpeas",
    "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate",
    "banana", "mango", "grapes", "watermelon", "muskmelon",
    "apple", "orange", "papaya", "coconut", "cotton", "jute", "coffee"
]

def analyze_errors(outputs_list, output_key, model_name):
    print(f"\n=== Error Analysis: {model_name} ===")
    errors = []

    for i, sample in enumerate(outputs_list):
        pred = sample[output_key].lower()
        ref_crop = sample["reference"].split(":")[1].split(".")[0].strip().lower()

        error_type = None

        # Check hallucination — crop not in known list
        hallucinated = not any(c in pred for c in KNOWN_CROPS)
        if hallucinated:
            error_type = "HALLUCINATION"

        # Check wrong crop
        elif ref_crop not in pred:
            error_type = "WRONG_CROP"

        # Check missing reasoning
        elif "reasoning" not in pred and "because" not in pred and "suitable" not in pred:
            error_type = "MISSING_REASONING"

        if error_type:
            errors.append({
                "index":      i,
                "error_type": error_type,
                "input":      sample["input"],
                "expected":   sample["reference"][:100],
                "predicted":  sample[output_key][:100]
            })

    print(f"Total errors: {len(errors)} / {len(outputs_list)}")
    for e in errors[:5]:  # show first 5
        print(f"\n[{e['error_type']}] Sample {e['index']}")
        print(f"  Input:     {e['input'][:80]}")
        print(f"  Expected:  {e['expected'][:80]}")
        print(f"  Predicted: {e['predicted'][:80]}")

    return errors

errors_b1 = analyze_errors(baseline_outputs, "baseline1_output", "Baseline 1")
errors_b2 = analyze_errors(baseline2_outputs, "baseline2_output", "Baseline 2")
errors_ft = analyze_errors(system_outputs,   "system_output",   "AgroMind FT+RAG")

NameError: name 'baseline_outputs' is not defined

In [14]:
# Deliberate failure case tests — for your presentation
print("=== DELIBERATE FAILURE CASE TESTS ===")

failure_cases = [
    {
        "name": "Conflicting inputs (high rainfall + sandy soil)",
        "input": "N: 20, P: 15, K: 20, Temperature: 38.0°C, Humidity: 90.0%, pH: 7.8, Rainfall: 280.0mm"
    },
    {
        "name": "Extreme values (very low N, very high temp)",
        "input": "N: 2, P: 5, K: 5, Temperature: 45.0°C, Humidity: 20.0%, pH: 4.5, Rainfall: 10.0mm"
    },
    {
        "name": "All values at midpoint (ambiguous conditions)",
        "input": "N: 50, P: 50, K: 50, Temperature: 25.0°C, Humidity: 60.0%, pH: 6.5, Rainfall: 100.0mm"
    },
    {
        "name": "Very high K, low rainfall (unusual combination)",
        "input": "N: 10, P: 40, K: 120, Temperature: 30.0°C, Humidity: 55.0%, pH: 7.0, Rainfall: 35.0mm"
    },
]

for fc in failure_cases:
    print(f"\n--- {fc['name']} ---")
    response, _ = agromind_pipeline(
        fc["input"], ft_model, tokenizer,
        use_rag=True, log_conn=db_conn
    )
    print(f"Input:  {fc['input']}")
    print(f"Output: {response}")

=== DELIBERATE FAILURE CASE TESTS ===

--- Conflicting inputs (high rainfall + sandy soil) ---


NameError: name 'agromind_pipeline' is not defined

## 12. Improvement Summary
### CRITERION vii — Clear improvement demonstration

In [15]:
import matplotlib
matplotlib.use("Agg")  # Windows-safe backend
import matplotlib.pyplot as plt
import numpy as np

metrics_names = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "Crop Accuracy"]
b1_vals = [metrics_b1["rouge1"], metrics_b1["rouge2"], metrics_b1["rougeL"], metrics_b1["bleu"], metrics_b1["crop_accuracy"]]
b2_vals = [metrics_b2["rouge1"], metrics_b2["rouge2"], metrics_b2["rougeL"], metrics_b2["bleu"], metrics_b2["crop_accuracy"]]
ft_vals = [metrics_ft["rouge1"], metrics_ft["rouge2"], metrics_ft["rougeL"], metrics_ft["bleu"], metrics_ft["crop_accuracy"]]

x     = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width, b1_vals, width, label="Baseline 1 (zero-shot)",  color="#E24B4A")
bars2 = ax.bar(x,         b2_vals, width, label="Baseline 2 (RAG only)",   color="#EF9F27")
bars3 = ax.bar(x + width, ft_vals, width, label="AgroMind (FT + RAG)",     color="#1D9E75")

ax.set_xlabel("Metric")
ax.set_ylabel("Score")
ax.set_title("AgroMind — Model Comparison Across All Metrics")
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)

for bar in [*bars1, *bars2, *bars3]:
    ax.annotate(f"{bar.get_height():.2f}",
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("evaluation_chart.png", dpi=150)
plt.close()
print("Chart saved to evaluation_chart.png")

NameError: name 'metrics_b1' is not defined

In [ ]:
# Print final improvement summary
print("\n" + "="*60)
print("         AGROMIND — IMPROVEMENT SUMMARY")
print("="*60)
print(f"{'Metric':<20} {'B1':>8} {'B2':>8} {'AgroMind':>10} {'Δ vs B1':>10}")
print("-"*60)

metric_map = [
    ("ROUGE-1",       "rouge1"),
    ("ROUGE-2",       "rouge2"),
    ("ROUGE-L",       "rougeL"),
    ("BLEU",          "bleu"),
    ("Crop Accuracy", "crop_accuracy"),
]

for name, key in metric_map:
    b1 = metrics_b1[key]
    b2 = metrics_b2[key]
    ft = metrics_ft[key]
    delta = ((ft - b1) / max(b1, 1e-9)) * 100
    print(f"{name:<20} {b1:>8.4f} {b2:>8.4f} {ft:>10.4f} {delta:>+9.1f}%")

print("="*60)
print("B1 = Baseline 1 (zero-shot)")
print("B2 = Baseline 2 (RAG only, no fine-tuning)")
print("AgroMind = Fine-tuned Gemma 2B-IT + RAG")

## 13. Save All Outputs

In [ ]:
import json

# Save all model outputs for analysis
all_results = []
for i, sample in enumerate(test_subset):
    all_results.append({
        "index":       i,
        "input":       sample["input"],
        "reference":   sample["reference"],
        "baseline1":   baseline_outputs[i]["baseline1_output"],
        "baseline2":   baseline2_outputs[i]["baseline2_output"],
        "agromind":    system_outputs[i]["system_output"],
    })

with open("all_model_outputs.json", "w") as f:
    json.dump(all_results, f, indent=2)

# Save metrics
with open("metrics_summary.json", "w") as f:
    json.dump({
        "baseline1": metrics_b1,
        "baseline2": metrics_b2,
        "agromind":  metrics_ft
    }, f, indent=2)

# Save error analysis
with open("error_analysis.json", "w") as f:
    json.dump({
        "baseline1_errors": errors_b1,
        "baseline2_errors": errors_b2,
        "agromind_errors":  errors_ft
    }, f, indent=2)

print("All outputs saved:")
print("  agromind_adapter/        — LoRA adapter weights")
print("  agromind_merged/         — merged full model (if run)")
print("  chromadb_store/          — vector database")
print("  agromind_logs.db         — SQLite query log")
print("  data/processed/          — train/val/test splits")
print("  evaluation_results.csv   — metric scores")
print("  evaluation_chart.png     — comparison chart")
print("  all_model_outputs.json   — all predictions")
print("  metrics_summary.json     — metrics summary")
print("  error_analysis.json      — error cases")

## 14. Criteria Checklist

| Criterion | Implementation | Location |
|-----------|---------------|----------|
| i. Dataset quality + split | Kaggle crop dataset, 2200 rows, 70/15/15 split | Cells 2–5 |
| ii. PEFT fine-tuning | QLoRA on Gemma 2B-IT, r=16, 4-bit NF4 | Cells 6–10 |
| iii. Baseline comparison | B1: zero-shot, B2: RAG-only, AgroMind: FT+RAG | Cells 3, 9 |
| iv. Data storage | ChromaDB (vector) + SQLite (relational) | Cells 11–12 |
| v. BLEU + ROUGE metrics | evaluate library, crop accuracy metric | Cell 15 |
| vi. Error + hallucination analysis | Error types: wrong crop, hallucination, missing reasoning | Cells 16–17 |
| vii. Real-world applicability | India-specific crops, Rajasthan/Punjab context, live demo input | Throughout |